In [1]:
# | default_exp schema_extractor

In [2]:
# | export
from sqlmodel import Session, select
from seo_rat.article import get_article_by_path, insert_article, Article
from seo_rat.sqlite_db import SQLiteDB
from seo_rat.models import Website
from seo_rat.content_mapper import map_all_urls_to_files,fetch_sitemap_urls

In [3]:
# | hide
#| eval: false
from dotenv import load_dotenv
import os

db = SQLiteDB()
load_dotenv()
# BASE_PATH = os.environ["SEO_RAT_BASE_PATH"]
BASE_PATH = os.environ["AWAZLY_PATH"]


In [4]:
#| export 
from seo_rat.gsc_storage import get_top_queries
from seo_rat.article import  get_articles_by_website
from seo_rat.gsc_client import get_date_range

## FAQ Extractor (Rule-Based, Arabic + English)

In [8]:
# | export
import re

_ARABIC_RE = re.compile(r"[\u0600-\u06FF]")

_EN_Q_WORDS = re.compile(
    r"^(what|who|why|when|where|which|how|can|is|are|do|does|did|will|should|"
    r"could|would|may|might|shall)\b",
    re.IGNORECASE,
)

# Require a space after ما/من/هل etc. to avoid matching nouns like ماكينة/مادة/مانع
_AR_Q_WORDS = re.compile(
    r"^("
    r"ما (هو|هي|هم|هي|الح|ال|يع|تع|يج|ه[ا-ي])|"  # ما هو / ما هي / ما ال...
    r"ماذا |"
    r"من أين |من اين |"
    r"أين |اين |"
    r"كيف |كيفية |كيفيه |"
    r"هل |"
    r"لماذا |"
    r"كم |"
    r"بماذا |"
    r"لمن |"
    r"متى |"
    r"أي |"
    r"من "
    r")"
)

def _is_arabic(text: str) -> bool:
    return len(_ARABIC_RE.findall(text)) > len(text) * 0.3

def _is_question(query: str) -> bool:
    q = query.strip()
    if q.endswith("?") or q.endswith("؟"):
        return True
    return bool(_AR_Q_WORDS.match(q) if _is_arabic(q) else _EN_Q_WORDS.match(q))

def extract_faq_queries(rows: list[dict] | list[str]) -> list[str]:
    """Filter GSC queries that are questions (Arabic or English).
    Accepts either raw query strings or dicts from get_top_queries."""
    queries = (r["query"] if isinstance(r, dict) else r for r in rows)
    return list(filter(_is_question, queries))


In [13]:
#| eval: false
#| hide
#| export
path = "https://awazly.com/malm-dhanat-aldwadmy"
website_name="https://awazly.com/"

start, end = get_date_range("last_days", days=90*8)
print(start, end)

with db.get_session() as session:
    articles = get_articles_by_website(session, website_id=1)
    for article in articles: 
        page_path= article.url.removeprefix(website_name)
        top_query = get_top_queries(session=session,site_url="sc-domain:awazly.com",start_date=start,end_date=end,page_path=page_path,limit=1000,sort_by="impressions")
        if  extract_faq_queries(top_query)==[]:
            continue
        print(extract_faq_queries(top_query))
        print(article.file_path)


2024-04-05 2026-03-26
['ما هي عيوب الدهان البلاستيك']
/home/kobo/Desktop/astro_hacking/awazl/src/content/post/معلم_دهانات_كفر_الشيخ.md
['كيف انظف الكنب المخمل من البقع', 'كيفية تنظيف الكنب المخمل', 'كيف انظف الكنب المخمل', 'كيفية تنظيف الكنب بدون ماء', 'كيف انظف الكنب من البقع']
/home/kobo/Desktop/astro_hacking/awazl/src/content/post/غسيل_كنب_بالدوادمي.md
['كم تبلغ تكلفة بناء منزل في السعودية']
/home/kobo/Desktop/astro_hacking/awazl/src/content/post/الخرسانة_المعزولة.md
['ما الحل عند تسرب الماء داخل الجدار', 'اين يباع جهاز كشف تسرب المياه', 'كيف اعرف مكان تسرب الماء تحت البلاط؟']
/home/kobo/Desktop/astro_hacking/awazl/src/content/post/كشف_تسربات_المياه_بمكة.md
['كم سعر عزل الفوم', 'كم سعر متر عزل الفوم']
/home/kobo/Desktop/astro_hacking/awazl/src/content/post/شركة_عزل_اسطح_بمكة.md
['ما هو البورديوم', 'ما هو البورديوم للمطابخ', 'ما هي خامة البورديوم', 'ما هي الالوان', 'كم سعره']
/home/kobo/Desktop/astro_hacking/awazl/src/content/post/بورديوم.md
['ما هو grc في البناء', 'ما هو الجي ار سي'

In [9]:
#| eval: false
#| hide
should_match = [
    # Arabic questions
    "ما هو البورديوم", "ما هي خامة البورديوم", "ما هو grc في البناء",
    "كيف انظف الكنب المخمل من البقع", "كيفية تنظيف الكنب المخمل",
    "هل اسطوانة الغاز تنفجر", "هل الايبوكسي افضل ام السيراميك",
    "كم سعر عزل الفوم", "كم وزن اسطوانة الغاز",
    "اين يباع جهاز كشف تسرب المياه", "من اين استطيع شراء انبوبة غاز جديده",
    "لماذا يهمنا سرعة الصفحة",
    # English questions
    "how to improve page speed", "what is a backlink", "is fiber gas cylinder safe",
]

should_not_match = [
    "ماكينة غسيل الكنب", "مادة grc", "مادة الايبوكسي", "منظف كنب مخمل",
    "مانع تسرب السوائل ساكو", "ماسورة غاز", "ماصورة غاز",
    "منازل جده", "مادة بناء", "منظم غاز", "best seo tools", "digital marketing",
]

print("✓ Should match (questions):")
for q in should_match:
    result = _is_question(q)
    mark = "✓" if result else "✗ MISSED"
    print(f"  {mark}  {q}")

print("\n✗ Should NOT match (not questions):")
for q in should_not_match:
    result = _is_question(q)
    mark = "✗ FALSE POSITIVE" if result else "✓"
    print(f"  {mark}  {q}")


✓ Should match (questions):
  ✓  ما هو البورديوم
  ✓  ما هي خامة البورديوم
  ✓  ما هو grc في البناء
  ✓  كيف انظف الكنب المخمل من البقع
  ✓  كيفية تنظيف الكنب المخمل
  ✓  هل اسطوانة الغاز تنفجر
  ✓  هل الايبوكسي افضل ام السيراميك
  ✓  كم سعر عزل الفوم
  ✓  كم وزن اسطوانة الغاز
  ✓  اين يباع جهاز كشف تسرب المياه
  ✓  من اين استطيع شراء انبوبة غاز جديده
  ✓  لماذا يهمنا سرعة الصفحة
  ✓  how to improve page speed
  ✓  what is a backlink
  ✓  is fiber gas cylinder safe

✗ Should NOT match (not questions):
  ✓  ماكينة غسيل الكنب
  ✓  مادة grc
  ✓  مادة الايبوكسي
  ✓  منظف كنب مخمل
  ✓  مانع تسرب السوائل ساكو
  ✓  ماسورة غاز
  ✓  ماصورة غاز
  ✓  منازل جده
  ✓  مادة بناء
  ✓  منظم غاز
  ✓  best seo tools
  ✓  digital marketing
